# Stage 1 — DBpedia query collection

**Pipeline:** **01_query_collection** → 02_triplet_mining → rebuild_dbpedia_clean_splits.py (see `datasets/DATASET.md`).

**Purpose:** fetch SELECT queries from the LSQ DBpedia log (without `COUNT` and `dbpo:wikipageid`), execute each on the DBpedia endpoint and keep those that return at least one row.

**Run from:** `datasets/scripts/`

**Outputs:**
- `../dbpedia-splits/sources/lsq_sample_dbpedia.csv` — all fetched LSQ query texts
- `../dbpedia-splits/sources/successful_queries_dbpedia.json` — non-empty queries (`[{"sparql_query": ...}]`), input of Stage 2

**Note:** LSQ is paged with `LIMIT`/`OFFSET` and no `ORDER BY`, so a rerun is not guaranteed to return the same queries. The released dataset is fixed by `../dbpedia-splits/unique_queries.json` (see `DATASET.md`).


### 1. Imports

In [ ]:
import json
import time
import pandas as pd
import requests
from tqdm import tqdm
from pathlib import Path
from SPARQLWrapper import SPARQLWrapper, JSON

SOURCES = Path("../dbpedia-splits/sources")
SOURCES.mkdir(parents=True, exist_ok=True)
LSQ_SAMPLE_CSV = SOURCES / "lsq_sample_dbpedia.csv"
SUCCESSFUL_JSON = SOURCES / "successful_queries_dbpedia.json"

### 2. LSQ endpoint and query fetch

In [ ]:
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
QUERY_TEMPLATE = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#> 
PREFIX sd: <http://www.w3.org/ns/sparql-service-description#> 

SELECT DISTINCT ?text
WHERE {{ 
   ?query lsqv:hasRemoteExec/lsqv:endpoint <https://dbpedia.org/sparql> .
   ?query lsqv:text ?text . 
   ?query lsqv:hasSpin ?spin .
   ?spin a <http://spinrdf.org/sp#Select> . 
   ?query lsqv:hasStructuralFeatures ?features .

   FILTER NOT EXISTS {{ ?features lsqv:usesFeature lsqv:Count . }}

   FILTER(!CONTAINS(LCASE(STR(?text)), "dbpo:wikipageid"))
}}
LIMIT 1000
OFFSET {offset}
"""

def run_lsq_query(offset):
    sparql = SPARQLWrapper(LSQ_ENDPOINT)
    sparql.setQuery(QUERY_TEMPLATE.format(offset=offset))
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    return [result["text"]["value"] for result in results["results"]["bindings"]]

def fetch_all_queries(num_batches=60, batch_size=1000, delay=0.5):
    all_results = []
    for i in range(num_batches):
        offset = i * batch_size
        print(f"Fetching batch {i+1} with OFFSET {offset}...")
        try:
            rows = run_lsq_query(offset)
            all_results.extend(rows)
        except Exception as e:
            print(f"Error at offset {offset}: {e}")
        time.sleep(delay)
    return all_results

### 3. Fetch, save to CSV, and load

In [ ]:
queries = fetch_all_queries()
pd.DataFrame(queries, columns=["text"]).to_csv(LSQ_SAMPLE_CSV, index=False, encoding="utf-8")
print(f"Results saved to {LSQ_SAMPLE_CSV}")

df = pd.read_csv(LSQ_SAMPLE_CSV)
print(f"Total queries loaded: {len(df)}")

### 4. Duplicate analysis

In [ ]:
total_rows = len(df)
unique_rows = df["text"].nunique()
duplicate_count = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Duplicate rows: {duplicate_count}")

duplicates = df[df.duplicated("text", keep=False)].sort_values("text")
print("\nSample duplicates:")
print(duplicates.head(10))

### 5. DBpedia endpoint helper

In [ ]:
def query_dbpedia(sparql_query):
    endpoint = "https://dbpedia.data.dice-research.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    try:
        response = requests.get(endpoint, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        return None

### 6. Filter successful queries and save

In [ ]:
successful_queries = []
failed = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Querying DBpedia"):
    sparql_query = row.get("text")
    if sparql_query:
        results = query_dbpedia(sparql_query)
        if results and "results" in results and results["results"]["bindings"]:
            successful_queries.append({"sparql_query": sparql_query})
        else:
            failed += 1

print(f"Successful queries: {len(successful_queries)}")
print(f"Failed queries: {failed}")

with open(SUCCESSFUL_JSON, "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)
print(f"Saved {len(successful_queries)} queries to {SUCCESSFUL_JSON}")
